# Phase 1: Inference and Panel Diagnostics

## The Hook
To accurately measure the impact of promotions across the Dominicks Finer Foods chain, we first need to understand the underlying structure of our data. Retail demand is inherently noisy and highly heterogeneous. 

## The Setup
We load the raw transactional data and product metadata to analyze cross-sectional variance (Store-to-Store differences) and distribution skewness. This validates our choice of **Fixed Effects** and **Log Transformations** in the regression pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- HIGH PRIORITY PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data

print(f"Project root verified: {project_root}")

# Load for Panel Analysis
transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
df = preprocess_data(transactions, products)
df.head()

## 1. Distribution Skewness: Why We Used Log Transformations
Retail sales and prices are notoriously right-skewed. If we ran OLS on raw units, the residuals would violate homoscedasticity assumptions. Let's look at the density of Sales to justify the `log1p` approach.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sample_df = df.sample(min(100000, len(df)))

# Raw Sales Distribution
sns.histplot(sample_df['Sales'], bins=50, ax=axes[0], color='#2E86AB')
axes[0].set_title('Raw Sales (Right-Skewed)')
axes[0].set_xlabel('Units Sold')

# Log Sales Distribution
sns.histplot(np.log1p(sample_df['Sales']), bins=50, ax=axes[1], color='#E63946')
axes[1].set_title('Log(Sales) (Normalized)')
axes[1].set_xlabel('Log(Units Sold)')

plt.tight_layout()
plt.show()

## 2. Unobserved Heterogeneity: Why We Used Fixed Effects
A promotion at a high-volume suburban store will yield completely different absolute numbers than the exact same promotion at a low-volume urban store. This variance requires us to use High-Dimensional Fixed Effects.

In [ ]:
top_stores = df['Store_ID'].value_counts().nlargest(15).index
df_sample = df[df['Store_ID'].isin(top_stores)].copy()

plt.figure(figsize=(14, 7))
sns.boxplot(data=df_sample, x='Store_ID', y='Sales', order=top_stores, palette='viridis', showfliers=False)
plt.title('Baseline Sales Variance Across Top 15 Stores')
plt.xlabel('Store ID')
plt.ylabel('Weekly Sales')
plt.xticks(rotation=45)
plt.show()